# 01 — Data Exploration

Initial exploration of the security alert triage dataset.
Run the pipeline first: `py -3.13 src/pipeline.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import matplotlib.pyplot as plt
import numpy as np
from src.config import load_config

cfg = load_config()
con = duckdb.connect(cfg['paths']['database'], read_only=True)

In [ ]:
# Dataset overview
con.execute("SELECT COUNT(*) AS total_alerts FROM fact_alerts").fetchdf()

In [ ]:
# Grade distribution
con.execute("""
    SELECT analyst_grade, COUNT(*) AS count,
           ROUND(COUNT(*)::DOUBLE / SUM(COUNT(*)) OVER () * 100, 2) AS pct
    FROM fact_alerts
    GROUP BY analyst_grade
    ORDER BY count DESC
""").fetchdf()

In [ ]:
# Timestamp range
con.execute("""
    SELECT MIN(alert_timestamp) AS first_alert,
           MAX(alert_timestamp) AS last_alert,
           DATEDIFF('day', MIN(alert_timestamp), MAX(alert_timestamp)) AS span_days
    FROM fact_alerts
""").fetchdf()

In [ ]:
# Top 10 detection rules by volume
con.execute("""
    SELECT detection_rule, alert_volume, tp_count, fp_count, precision
    FROM dim_rule_performance
    ORDER BY alert_volume DESC
    LIMIT 10
""").fetchdf()

In [ ]:
# Entity type distribution
con.execute("""
    SELECT entity_type, alert_volume, precision, fp_rate
    FROM agg_entity_type
    ORDER BY alert_volume DESC
""").fetchdf()

In [ ]:
# MITRE technique distribution
con.execute("""
    SELECT mitre_technique, alert_volume, precision, fp_burden_share
    FROM agg_mitre
    WHERE mitre_technique != 'unknown'
    ORDER BY alert_volume DESC
    LIMIT 15
""").fetchdf()

In [ ]:
con.close()